# Fruletov Audio EDA

In [ ]:
from pathlib import Path

import gdown


drive_folder_url = "https://drive.google.com/drive/folders/11gBa3XlAIdDyq4p3n2hnFBP6uskMtm-_"
output_dir = Path("../data/raw/Fruletov")
dataset_dir = output_dir / "Dataset Nov 2021"

if dataset_dir.exists() and list(dataset_dir.glob("*.wav")):
    print(f"Dataset already exists: {dataset_dir}")
    print(f"Found {len(list(dataset_dir.glob('*.wav')))} wav files")
else:
    output_dir.mkdir(parents=True, exist_ok=True)
    gdown.download_folder(
        url=drive_folder_url,
        output=str(output_dir),
        quiet=False,
        use_cookies=False,
    )
    print(f"Downloaded to: {dataset_dir}")
    print(f"Found {len(list(dataset_dir.glob('*.wav')))} wav files")

Исселдуем набор данных чтобы понять, какие там есть данные

In [ ]:
from pathlib import Path
output_dir = Path("../data/raw/Fruletov")
dataset_dir = output_dir / "Dataset Nov 2021"

In [ ]:
from collections import Counter
import wave
import pandas as pd

def get_wav_file_overview(wav_dir: Path):
    """
    Выводит базовую информацию о .wav файлах в директории:
    - количество файлов
    - примеры имен
    - уникальные расширения
    - распределение по sample rate, числу каналов, длительности
    - суммарная длительность аудиофайлов
    Также выводит DataFrame-манифест с колонками:
      1. audio_path (относительный)
      2. file_name (без слова 'Full')
      3. sr файла
      4. duration в секундах
    """
    wav_paths = sorted(wav_dir.glob("*.wav"))
    print(f"Всего .wav файлов: {len(wav_paths)}")
    print("Примеры файлов:", [p.name for p in wav_paths[:5]])
    extensions = {p.suffix for p in wav_paths}
    print("Уникальные расширения:", extensions)
    
    sample_rates = []
    nchannels = []
    durations = []
    errors = []
    manifest_rows = []

    for p in wav_paths:
        try:
            with wave.open(str(p), "rb") as wf:
                sr = wf.getframerate()
                nc = wf.getnchannels()
                n_frames = wf.getnframes()
                duration = n_frames / sr
                sample_rates.append(sr)
                nchannels.append(nc)
                durations.append(duration)
                
 
                try:
                    rel_audio_path = p.relative_to(Path.cwd())
                except ValueError:
                    rel_audio_path = p.relative_to(wav_dir)
                file_name = p.name.replace("Full", "").replace("  ", " ").replace(".wav", "").strip()
                manifest_rows.append({
                    "audio_path": "../data/raw/Fruletov/Dataset Nov 2021/" + str(rel_audio_path),
                    "file_name": file_name,
                    "sr": sr,
                    "duration": duration,
                })
        except Exception as e:
            errors.append((p.name, str(e)))

    print("Уникальные sample rate:", sorted(set(sample_rates)))
    print("Уникальные числа каналов:", sorted(set(nchannels)))
    print("Распределение sample rate:", Counter(sample_rates))
    print("Распределение каналов:", Counter(nchannels))
    if durations:
        print(f"Минимальная длительность: {min(durations):.2f} c")
        print(f"Максимальная длительность: {max(durations):.2f} c")
        print(f"Средняя длительность: {sum(durations)/len(durations):.2f} c")
        print(f"Суммарная длительность аудиофайлов: {sum(durations)/3600:.2f} часов ({sum(durations):.0f} секунд)")
    if errors:
        print(f"\nОшибки при чтении {len(errors)} файлов:")
        for name, err in errors:
            print(f"{name}: {err}")

    if manifest_rows:
        manifest_df = pd.DataFrame(manifest_rows, columns=["audio_path", "file_name", "sr", "duration"])
        print("\nDataFrame-манифест по директории:")
        display(manifest_df)

get_wav_file_overview(dataset_dir)

In [ ]:
import IPython.display as ipd

def listen_wav_example(wav_dir: Path, index: int = 0):
    """
    Воспроизводит i-ю .wav аудиозапись в директории с помощью IPython.display.
    """
    wav_paths = sorted(wav_dir.glob("*.wav"))
    if not wav_paths:
        print("В директории нет .wav файлов.")
        return
    example_path = wav_paths[index]
    print(f"Воспроизводится файл: {example_path.name}")
    display(ipd.Audio(str(example_path)))

listen_wav_example(dataset_dir)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

def plot_wav_waveform_seaborn(wav_dir: Path, index: int = 0):
    """
    Рисует волны (амплитуда от времени) двух каналов i-й .wav аудиозаписи
    Каждый канал - отдельный график.
    """
    wav_paths = sorted(wav_dir.glob("*.wav"))
    if not wav_paths:
        print("В директории нет .wav файлов.")
        return
    example_path = wav_paths[index]
    data, sr = sf.read(str(example_path))
    if data.ndim == 1:
        print("Файл одноканальный, рисую только один канал.")
        data = np.expand_dims(data, axis=1)
    n_channels = data.shape[1]
    duration = len(data) / sr
    times = np.linspace(0, duration, num=len(data))
    for chan in range(n_channels):
        plt.figure(figsize=(14, 3))
        sns.lineplot(x=times, y=data[:, chan])
        plt.title(f"Волна: {example_path.name} (канал {chan+1})")
        plt.xlabel("Время, с")
        plt.ylabel("Амплитуда")
        plt.tight_layout()
        plt.show()

plot_wav_waveform_seaborn(dataset_dir)

# Идеи
- 1-3 с фреймы
- нормализация
- ресемплинг 16к
- Объединение в 1 канал
 